# Stage 6: All-Exercises Counting Baseline

This notebook trains counting-only TCN baselines on the normalized pose sequences from Stage 5.
It keeps the task focused on repetition counting by running one baseline per exercise first.


In [ ]:
from google.colab import drive
drive.mount('/content/drive')


In [ ]:
from pathlib import Path
import shutil

CODE_ROOT = Path('/content/CV_Image_pose_detection')
DRIVE_PROJECT_ROOT = Path('/content/drive/MyDrive/FinalProjectCV/CV_Image_pose_detection')

TRAINER_REL = Path('artifacts/3_Modeling/train_pose_count_tcn.py')
COMPARE_REL = Path('artifacts/3_Modeling/compare_count_run_to_baseline.py')
TRAINER_SRC = CODE_ROOT / TRAINER_REL
TRAINER_DST = DRIVE_PROJECT_ROOT / TRAINER_REL
COMPARE_SRC = CODE_ROOT / COMPARE_REL
COMPARE_DST = DRIVE_PROJECT_ROOT / COMPARE_REL

if TRAINER_SRC.exists():
    TRAINER_DST.parent.mkdir(parents=True, exist_ok=True)
    shutil.copy2(TRAINER_SRC, TRAINER_DST)

if COMPARE_SRC.exists():
    COMPARE_DST.parent.mkdir(parents=True, exist_ok=True)
    shutil.copy2(COMPARE_SRC, COMPARE_DST)

ANNOTATION_DIR = DRIVE_PROJECT_ROOT / 'Data/LLSP/annotation_cleaned'
SEQUENCE_INDEX = ANNOTATION_DIR / 'pose_sequence_index.csv'

print('DRIVE_PROJECT_ROOT =', DRIVE_PROJECT_ROOT)
print('TRAINER_DST =', TRAINER_DST)
print('COMPARE_DST =', COMPARE_DST)
print('SEQUENCE_INDEX exists =', SEQUENCE_INDEX.exists())


In [ ]:
import pandas as pd

seq_df = pd.read_csv(SEQUENCE_INDEX)
display(seq_df.head())

counts_df = (
    seq_df.groupby(['type', 'split']).size().unstack(fill_value=0).sort_index()
)
display(counts_df)

available_exercises = counts_df[(counts_df.get('train', 0) > 0) & (counts_df.get('valid', 0) > 0)].index.tolist()
print('Available exercises with both train and valid rows:', available_exercises)


In [ ]:
SEQ_LEN = 192
EPOCHS = 80
BATCH_SIZE = 16
LR = 1e-3
WEIGHT_DECAY = 1e-4
CHANNELS = 96
KERNEL_SIZE = 3
NUM_BLOCKS = 4
DROPOUT = 0.2
PATIENCE = 15
LOSS = 'l1'
EVAL_TRANSFORM = 'raw'
SELECTION_METRIC = 'mae'
SAMPLER = 'balanced_count'
TIME_WARP_RANGE = 0.12
FEATURE_NOISE_STD = 0.02
FRAME_DROPOUT_PROB = 0.03

EXERCISES = available_exercises

RUNS = []
for exercise in EXERCISES:
    safe_name = exercise.replace('-', '_')
    RUNS.append({
        'run_name': f'pose_count_tcn_{safe_name}',
        'exercise': exercise,
        'seq_len': SEQ_LEN,
        'epochs': EPOCHS,
        'batch_size': BATCH_SIZE,
        'lr': LR,
        'weight_decay': WEIGHT_DECAY,
        'channels': CHANNELS,
        'kernel_size': KERNEL_SIZE,
        'num_blocks': NUM_BLOCKS,
        'dropout': DROPOUT,
        'patience': PATIENCE,
        'loss': LOSS,
        'eval_transform': EVAL_TRANSFORM,
        'selection_metric': SELECTION_METRIC,
        'sampler': SAMPLER,
        'time_warp_range': TIME_WARP_RANGE,
        'feature_noise_std': FEATURE_NOISE_STD,
        'frame_dropout_prob': FRAME_DROPOUT_PROB,
    })

pd.DataFrame(RUNS)


In [ ]:
import subprocess
import pandas as pd

training_failures = []
failed_run_names = []
for cfg in RUNS:
    cmd = [
        'python', str(TRAINER_DST),
        '--project-dir', str(DRIVE_PROJECT_ROOT),
        '--index-csv', str(SEQUENCE_INDEX),
        '--run-name', cfg['run_name'],
        '--exercise', cfg['exercise'],
        '--seq-len', str(cfg['seq_len']),
        '--epochs', str(cfg['epochs']),
        '--batch-size', str(cfg['batch_size']),
        '--lr', str(cfg['lr']),
        '--weight-decay', str(cfg['weight_decay']),
        '--channels', str(cfg['channels']),
        '--kernel-size', str(cfg['kernel_size']),
        '--num-blocks', str(cfg['num_blocks']),
        '--dropout', str(cfg['dropout']),
        '--patience', str(cfg['patience']),
        '--loss', cfg['loss'],
        '--eval-transform', cfg['eval_transform'],
        '--selection-metric', cfg['selection_metric'],
        '--sampler', cfg['sampler'],
        '--time-warp-range', str(cfg['time_warp_range']),
        '--feature-noise-std', str(cfg['feature_noise_std']),
        '--frame-dropout-prob', str(cfg['frame_dropout_prob']),
        '--device', 'cuda',
    ]
    print('\nRunning:', ' '.join(cmd))
    try:
        subprocess.run(cmd, check=True)
    except subprocess.CalledProcessError as exc:
        training_failures.append({
            'exercise': cfg['exercise'],
            'run_name': cfg['run_name'],
            'returncode': exc.returncode,
        })
        failed_run_names.append(cfg['run_name'])
        print(f"FAILED: {cfg['exercise']} (returncode={exc.returncode})")
        print('Use the debug cell below to print the full traceback for this run.')

if training_failures:
    display(pd.DataFrame(training_failures))
else:
    print('All runs completed successfully.')


In [ ]:
DEBUG_RUN_NAME = failed_run_names[0] if failed_run_names else None
debug_cfg = next((cfg for cfg in RUNS if cfg['run_name'] == DEBUG_RUN_NAME), None)

if debug_cfg is None:
    print('No failed run available to debug.')
else:
    cmd = [
        'python', str(TRAINER_DST),
        '--project-dir', str(DRIVE_PROJECT_ROOT),
        '--index-csv', str(SEQUENCE_INDEX),
        '--run-name', debug_cfg['run_name'],
        '--exercise', debug_cfg['exercise'],
        '--seq-len', str(debug_cfg['seq_len']),
        '--epochs', str(debug_cfg['epochs']),
        '--batch-size', str(debug_cfg['batch_size']),
        '--lr', str(debug_cfg['lr']),
        '--weight-decay', str(debug_cfg['weight_decay']),
        '--channels', str(debug_cfg['channels']),
        '--kernel-size', str(debug_cfg['kernel_size']),
        '--num-blocks', str(debug_cfg['num_blocks']),
        '--dropout', str(debug_cfg['dropout']),
        '--patience', str(debug_cfg['patience']),
        '--loss', debug_cfg['loss'],
        '--eval-transform', debug_cfg['eval_transform'],
        '--selection-metric', debug_cfg['selection_metric'],
        '--sampler', debug_cfg['sampler'],
        '--time-warp-range', str(debug_cfg['time_warp_range']),
        '--feature-noise-std', str(debug_cfg['feature_noise_std']),
        '--frame-dropout-prob', str(debug_cfg['frame_dropout_prob']),
        '--device', 'cuda',
    ]
    print('\nDebugging:', ' '.join(cmd))
    result = subprocess.run(cmd, text=True, capture_output=True)
    if result.stdout:
        print('\n[stdout]\n')
        print(result.stdout)
    if result.stderr:
        print('\n[stderr]\n')
        print(result.stderr)
    print('returncode =', result.returncode)
    if result.returncode == 0:
        print('Debug run completed successfully.')


In [ ]:
import json
import pandas as pd

rows = []
for cfg in RUNS:
    metrics_path = DRIVE_PROJECT_ROOT / 'artifacts/3_Modeling/training_outputs' / cfg['run_name'] / 'metrics_summary.json'
    if not metrics_path.exists():
        continue
    with open(metrics_path, 'r', encoding='utf-8') as f:
        metrics = json.load(f)
    rows.append({
        'exercise': cfg['exercise'],
        'run_name': cfg['run_name'],
        'best_epoch': metrics.get('best_epoch'),
        'valid_mae': metrics['valid_metrics']['mae'],
        'valid_rmse': metrics['valid_metrics']['rmse'],
        'valid_within_1': metrics['valid_metrics']['within_1'],
    })

results_df = pd.DataFrame(rows)
if results_df.empty:
    print('No metrics_summary.json files found yet.')
else:
    display(results_df.sort_values('exercise'))
    macro = results_df[['valid_mae', 'valid_rmse', 'valid_within_1']].mean().to_frame('macro_avg').T
    display(macro)


In [ ]:
import subprocess
import json
import pandas as pd

baseline_rows = []
comparison_failures = []
for cfg in RUNS:
    run_dir = DRIVE_PROJECT_ROOT / 'artifacts/3_Modeling/training_outputs' / cfg['run_name']
    pred_path = run_dir / 'predictions.csv'
    if not pred_path.exists():
        continue
    cmd = [
        'python', str(COMPARE_DST),
        '--index-csv', str(SEQUENCE_INDEX),
        '--predictions-csv', str(pred_path),
        '--exercise', cfg['exercise'],
    ]
    print('\nComparing:', ' '.join(cmd))
    try:
        subprocess.run(cmd, check=True)
    except subprocess.CalledProcessError as exc:
        comparison_failures.append({
            'exercise': cfg['exercise'],
            'run_name': cfg['run_name'],
            'returncode': exc.returncode,
        })
        print(f"COMPARE FAILED: {cfg['exercise']} (returncode={exc.returncode})")
        continue
    summary_path = run_dir / 'baseline_comparison_summary.json'
    with open(summary_path, 'r', encoding='utf-8') as f:
        summary = json.load(f)
    baseline_rows.append({
        'exercise': cfg['exercise'],
        'run_name': cfg['run_name'],
        'model_mae': summary['model_metrics']['mae'],
        'baseline_mae': summary['baseline_metrics']['mae'],
        'delta_mae': summary['delta_vs_baseline']['mae'],
        'model_within_1': summary['model_metrics']['within_1'],
        'baseline_within_1': summary['baseline_metrics']['within_1'],
        'delta_within_1': summary['delta_vs_baseline']['within_1'],
        'model_beats_baseline': summary['row_level']['model_beats_baseline'],
        'valid_rows': summary['row_level']['valid_rows'],
    })

baseline_df = pd.DataFrame(baseline_rows)
if baseline_df.empty:
    print('No baseline comparison summaries found yet.')
else:
    display(baseline_df.sort_values('exercise'))

if comparison_failures:
    display(pd.DataFrame(comparison_failures))


In [ ]:
RUN_NAME = RUNS[0]['run_name'] if RUNS else None
if RUN_NAME is None:
    print('No configured runs available.')
else:
    pred_path = DRIVE_PROJECT_ROOT / 'artifacts/3_Modeling/training_outputs' / RUN_NAME / 'predictions.csv'
    pred_df = pd.read_csv(pred_path)
    display(pred_df[pred_df['split'] == 'valid'].sort_values('abs_error', ascending=False).head(20))
